In [22]:
!uv pip list | grep deepeval
!uv pip list | grep langchain

Using Python 3.12.3 environment at: /home/vsc/LLM_TUNE/QA-FineTune/venv/venv-RAG
deepeval                                 3.7.0
Using Python 3.12.3 environment at: /home/vsc/LLM_TUNE/QA-FineTune/venv/venv-RAG
langchain                                1.2.10
langchain-classic                        1.0.1
langchain-community                      0.4.1
langchain-core                           1.2.17
langchain-huggingface                    1.2.1
langchain-openai                         0.1.7
langchain-text-splitters                 1.1.1


In [2]:
import inspect
from deepeval.synthesizer.config import EvolutionConfig, Evolution

# Evolution enum 전체 값 확인
print("=== Evolution 옵션 ===")
for e in Evolution:
    print(f"  {e.name}: {e.value}")

=== Evolution 옵션 ===
  REASONING: Reasoning
  MULTICONTEXT: Multi-context
  CONCRETIZING: Concretizing
  CONSTRAINED: Constrained
  COMPARATIVE: Comparative
  HYPOTHETICAL: Hypothetical
  IN_BREADTH: In-Breadth


In [3]:
# EvolutionConfig 시그니처 확인
print("\n=== EvolutionConfig 시그니처 ===")
print(inspect.signature(EvolutionConfig))


=== EvolutionConfig 시그니처 ===
(num_evolutions: int = 1, evolutions: Dict[deepeval.synthesizer.types.Evolution, float] = <factory>) -> None


In [4]:
# EvolutionConfig 소스코드 확인
print("\n=== EvolutionConfig 소스 ===")
print(inspect.getsource(EvolutionConfig))


=== EvolutionConfig 소스 ===
@dataclass
class EvolutionConfig:
    num_evolutions: int = 1
    evolutions: Dict[Evolution, float] = field(
        default_factory=lambda: {
            Evolution.REASONING: 1 / 7,
            Evolution.MULTICONTEXT: 1 / 7,
            Evolution.CONCRETIZING: 1 / 7,
            Evolution.CONSTRAINED: 1 / 7,
            Evolution.COMPARATIVE: 1 / 7,
            Evolution.HYPOTHETICAL: 1 / 7,
            Evolution.IN_BREADTH: 1 / 7,
        }
    )



In [5]:
from deepeval.synthesizer import synthesizer as synth_module

# _evolve_input 또는 evolution 관련 메서드 소스 확인
print(inspect.getsource(synth_module.Synthesizer._a_evolve_input))

    async def _a_evolve_input(
        self,
        input: str,
        num_evolutions: int,
        evolutions: Dict[Union[Evolution, PromptEvolution], float],
        context: Optional[List[str]] = None,
        progress: Optional[Progress] = None,
        pbar_evolve_input_id: Optional[int] = None,
        remove_pbar: bool = True,
    ) -> Tuple[str, List[Union[Evolution, PromptEvolution]]]:
        evolved_input = input
        evolutions_used = []
        for _ in range(num_evolutions):
            # Randomize Evolution
            evolution_type = random.choices(
                list(evolutions.keys()), list(evolutions.values())
            )[0]

            # Create Evolution Prompt
            if isinstance(evolution_type, Evolution):
                evolution_method = evolution_map[evolution_type.value]
                prompt = evolution_method(input=evolved_input, context=context)
            elif isinstance(evolution_type, PromptEvolution):
                evolution_method

In [7]:
import os
import deepeval

# deepeval 패키지 경로
deepeval_path = os.path.dirname(deepeval.__file__)
synthesizer_path = os.path.join(deepeval_path, "synthesizer")

# synthesizer 폴더 안에 뭐가 있는지
print("=== synthesizer 폴더 내용 ===")
for f in os.listdir(synthesizer_path):
    print(f)

=== synthesizer 폴더 내용 ===
templates
schema.py
__init__.py
types.py
synthesizer.py
config.py
chunking
base_synthesizer.py
utils.py
__pycache__


In [8]:
# template 관련 파일 소스 읽기
template_path = os.path.join(synthesizer_path, "templates")  # 혹은 template.py
if os.path.isdir(template_path):
    for f in os.listdir(template_path):
        print(f)
else:
    # 직접 파일 열기
    with open(os.path.join(synthesizer_path, "template.py"), encoding="utf-8") as f:
        print(f.read())

__init__.py
template_prompt.py
template.py
__pycache__
template_extraction.py


In [9]:
import inspect
from deepeval.synthesizer.synthesizer import Synthesizer

# evolution_method 부분이 핵심 - MULTICONTEXT가 context를 어떻게 쓰는지
src = inspect.getsource(Synthesizer._a_evolve_input)
print(src)

    async def _a_evolve_input(
        self,
        input: str,
        num_evolutions: int,
        evolutions: Dict[Union[Evolution, PromptEvolution], float],
        context: Optional[List[str]] = None,
        progress: Optional[Progress] = None,
        pbar_evolve_input_id: Optional[int] = None,
        remove_pbar: bool = True,
    ) -> Tuple[str, List[Union[Evolution, PromptEvolution]]]:
        evolved_input = input
        evolutions_used = []
        for _ in range(num_evolutions):
            # Randomize Evolution
            evolution_type = random.choices(
                list(evolutions.keys()), list(evolutions.values())
            )[0]

            # Create Evolution Prompt
            if isinstance(evolution_type, Evolution):
                evolution_method = evolution_map[evolution_type.value]
                prompt = evolution_method(input=evolved_input, context=context)
            elif isinstance(evolution_type, PromptEvolution):
                evolution_method

In [10]:
import inspect
from deepeval.synthesizer.synthesizer import Synthesizer, evolution_map

print("=== evolution_map 키 ===")
for key, func in evolution_map.items():
    print(f"\n--- {key} ---")
    print(inspect.getsource(func))

=== evolution_map 키 ===

--- Reasoning ---
    @staticmethod
    def reasoning_evolution(input, context):
        return (
            EvolutionTemplate.base_instruction
            + f"""
            1. If `Input` can be solved with just a few simple thinking processes, you can rewrite it to explicitly request multiple-step reasoning.
            2. `Rewritten Input` should require readers to make multiple logical connections or inferences.
            3. `Rewritten Input` should be concise and understandable by humans.
            4. `Rewritten Input` should not contain phrases like  'based on the provided context' or 'according to the context'.
            5. `Rewritten Input` must be fully answerable from information in `Context`. 
            6. `Rewritten Input` should not contain more than 15 words. Use abbreviation wherever possible.

            **
            EXAMPLES

            Example context:
            Chlorophyll allows plants to absorb energy from light, and this ene

# 모든 템플릿 공통
Context:
{context}   # ← List[str]을 str()로 변환해서 그냥 넣음
```

즉 **모든 Evolution이 context 구조를 동일하게 취급**합니다. MULTICONTEXT만 특별히 다르게 처리하지 않습니다.

## 각 Evolution 분석

| Evolution | 핵심 지시 | context 필요성 |
|---|---|---|
| **REASONING** | 여러 논리적 추론 단계 요구 | 있으면 활용 |
| **MULTICONTEXT** | **context의 모든 요소를 사용해야 답 가능**하도록 | ⭐ 여러 개일수록 효과적 |
| **CONCRETIZING** | 일반 개념 → 구체적 개념으로 | 있으면 활용 |
| **CONSTRAINED** | 조건/제약 추가 | 있으면 활용 |
| **COMPARATIVE** | 두 개 이상 비교 | 있으면 활용 |
| **HYPOTHETICAL** | 가상 시나리오 추가 | 있으면 활용 |
| **IN_BREADTH** | 같은 도메인의 새로운 질문 생성 | 거의 무관 |

## MULTICONTEXT 프롬프트 예시가 답을 줌
```
Example context:
["Vaccines introduce a weakened...", "This exposure helps the immune system..."]
# ↑ 리스트에 여러 문자열 → 두 문장을 모두 알아야 답 가능한 질문 생성

In [1]:
import os
import nest_asyncio
nest_asyncio.apply()

In [2]:
import pandas as pd

XLSX_PATH = "/home/vsc/LLM_TUNE/QA-FineTune/main/data/고양시도서관 FAQ1.xlsx"
TXT_PATH  = "/home/vsc/LLM_TUNE/QA-FineTune/main/data/고양시도서관_FAQ1.txt"

df = pd.read_excel(XLSX_PATH)
print("컬럼:", df.columns.tolist())
print(f"행 수: {len(df)}")
# df.head(3)   # 주피터라면 표시

# 실제 컬럼명으로 바꾸세요 (당신의 파일 기준으로)
TITLE = df.columns[1]   # 예: '질문' 또는 '제목'
DES   = df.columns[2]   # 예: '답변' 또는 '내용'

with open(TXT_PATH, "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        q = str(row[TITLE]).strip()
        a = str(row[DES]).strip()
        if q and a and q != "nan" and a != "nan":
            f.write(f"Q: {q}\nA: {a}\n\n")

# 파일 확인
with open(TXT_PATH, encoding="utf-8") as f:
    content = f.read()
print(f"저장 완료: {TXT_PATH}")
print(f"파일 크기: {len(content)} 글자")
print("-" * 40)
print(content[:400])

컬럼: ['FAQ', 'TITLE', 'DES']
행 수: 110
저장 완료: /home/vsc/LLM_TUNE/QA-FineTune/main/data/고양시도서관_FAQ1.txt
파일 크기: 20562 글자
----------------------------------------
Q: 회원증을 대리발급 할 수 있나요?
A: 회원증 대리발급은 만14세 미만 아동, 만65세 이상 어르신, 장애인, 임산부만 가능하며 아래 구비서류를 지참하여 대리인이 방문 시 대리발급이 가능합니다.

· 대상 : 만14세 미만 아동, 만65세 이상 어르신, 장애인, 임산부
· 구비서류
  - 공통 : ①위임자 신분증, ②피위임자(대리인) 신분증
  - 장애인 : 장애인 복지카드 또는 장애인 증명서
  - 임신부 : 산모수첩 / 산모 : 주민등록등본(출산 후 12개월까지)
· 방법 : 홈페이지 회원가입 후 위 항목의 해당하는 구비서류를 지참하여 피위임자(대리인)이 도서관 방문

Q: 가족회원이 무엇인가요?
A: 가족회원이란, 고양시 도서관 정회원으로 가입 후 가족회원으로 등록 시 가족명의로 도서대출·도서예약


In [3]:
import re
import json
import logging
from deepeval.models import DeepEvalBaseLLM, DeepEvalBaseEmbeddingModel
from typing import List
from openai import OpenAI, AsyncOpenAI

logger = logging.getLogger(__name__)  # ✅ 여기서 선언

class VLLMModel(DeepEvalBaseLLM):
    def __init__(self, model_name: str, base_url: str):
        self.model_name = model_name
        self.client = OpenAI(base_url=base_url, api_key="vllm-dummy")
        self.async_client = AsyncOpenAI(base_url=base_url, api_key="vllm-dummy")

    def load_model(self): return self.client
    def get_model_name(self): return self.model_name

    def _is_response_schema(self, schema) -> bool:
        if schema is None:
            return False
        try:
            fields = schema.model_fields if hasattr(schema, 'model_fields') else {}
            return list(fields.keys()) == ['response']
        except Exception:
            return False

    def _get_system_prompt(self, schema) -> str:
        base = "모든 답변은 반드시 한국어로 작성하세요. "
        if schema is not None and self._is_response_schema(schema):
            return base + "간결하고 직접적으로 답변하세요."
        elif schema is not None:
            return base + "반드시 유효한 JSON 형식으로만 응답하세요. 설명이나 마크다운을 포함하지 마세요."
        else:
            return base + "도움이 되는 어시스턴트입니다."

    def _safe_json_parse(self, content: str, schema=None):
        if schema is not None and self._is_response_schema(schema):
            try:
                match = re.search(r'\{.*\}', content, re.DOTALL)
                if match:
                    data = json.loads(match.group(0))
                    if 'response' in data:
                        return schema(response=str(data['response']))
            except Exception:
                pass
            clean_text = content.strip().strip('"').strip("'")
            return schema(response=clean_text)

        try:
            match = re.search(r"(\{.*\}|\[.*\])", content, re.DOTALL)
            if not match:
                raise ValueError("JSON 구조를 찾을 수 없습니다.")

            json_str = match.group(1).strip()
            open_braces = json_str.count('{')
            close_braces = json_str.count('}')
            if open_braces > close_braces:
                json_str += '}' * (open_braces - close_braces)
                json_str = json_str.rstrip()
                if json_str.endswith(','): json_str = json_str[:-1]
                if ']' not in json_str and '[' in json_str: json_str += ']}' 

            data = json.loads(json_str)
            if schema:
                return schema(**data)
            return data

        except Exception as e:
            logger.warning(f"파싱 실패: {e} | 응답 요약: {content[:50]}...")
            if schema:
                try: return schema(data=[])
                except: return None
            return {"data": []}

    def generate(self, prompt: str, schema=None, **kwargs):
        res = self.client.chat.completions.create(
            model=self.model_name,
            messages=[
                {"role": "system", "content": self._get_system_prompt(schema)},  # ✅
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            max_tokens=3000
        )
        return self._safe_json_parse(res.choices[0].message.content, schema)

    async def a_generate(self, prompt: str, schema=None, **kwargs):
        res = await self.async_client.chat.completions.create(
            model=self.model_name,
            messages=[
                {"role": "system", "content": self._get_system_prompt(schema)},  # ✅
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            max_tokens=3000
        )
        return self._safe_json_parse(res.choices[0].message.content, schema)


class VLLMEmbedding(DeepEvalBaseEmbeddingModel):
    def __init__(self, model_name: str, base_url: str):
        self.model_name = model_name
        self.client = OpenAI(base_url=base_url, api_key="vllm-dummy")
        self.async_client = AsyncOpenAI(base_url=base_url, api_key="vllm-dummy")  # ✅ 추가

    def load_model(self): return self.client
    def get_model_name(self): return self.model_name

    def embed_text(self, text: str) -> List[float]:
        resp = self.client.embeddings.create(input=text, model=self.model_name)
        return resp.data[0].embedding

    def embed_texts(self, texts: List[str]) -> List[List[float]]:
        resp = self.client.embeddings.create(input=texts, model=self.model_name)
        return [d.embedding for d in resp.data]

    async def a_embed_text(self, text: str) -> List[float]:  # ✅ 수정
        resp = await self.async_client.embeddings.create(
            input=text, model=self.model_name
        )
        return resp.data[0].embedding

    async def a_embed_texts(self, texts: List[str]) -> List[List[float]]:  # ✅ 수정
        resp = await self.async_client.embeddings.create(
            input=texts, model=self.model_name
        )
        return [d.embedding for d in resp.data]

In [4]:
# ────────────────────────────────────────────────
# 인스턴스 생성 (당신의 vLLM 포트 그대로)
local_llm = VLLMModel(
    model_name="/models/Exaone-3.5-32B-Instruct",
    base_url="http://localhost:8002/v1"
)

local_embedder = VLLMEmbedding(
    model_name="/embeddings/dragonkue/snowflake-arctic-embed-l-v2.0-ko",
    base_url="http://localhost:8003/v1"
)

# print(local_llm.generate("ㅎㅇ"))

# 단일 텍스트 임베딩 테스트
single_embedding = local_embedder.embed_text("회원증을 대리발급 할 수 있나요?")
print("단일 임베딩 길이:", len(single_embedding))
print("첫 5개 값:", single_embedding[:5])  # 숫자 배열이 나오면 성공

# 여러 텍스트 배치 테스트
batch_embeddings = local_embedder.embed_texts([
    "회원증을 대리발급 할 수 있나요?",
    "가족회원이 무엇인가요?",
    "도서 대출 기간은 얼마나 되나요?"
])
print("배치 결과 개수:", len(batch_embeddings))
print("각 임베딩 길이:", [len(e) for e in batch_embeddings])

단일 임베딩 길이: 1024
첫 5개 값: [0.015615975484251976, -0.06612725555896759, -0.003962401766330004, -0.010326542891561985, -0.03276457637548447]
배치 결과 개수: 3
각 임베딩 길이: [1024, 1024, 1024]


In [5]:
import os
from openai import OpenAI, AsyncOpenAI
from deepeval.models import DeepEvalBaseLLM, DeepEvalBaseEmbeddingModel
from deepeval.synthesizer import Synthesizer
from deepeval.synthesizer.config import ContextConstructionConfig
from typing import List


synthesizer = Synthesizer(
    model=local_llm
)

with open(TXT_PATH, encoding="utf-8") as f:
    full_text = f.read().strip()  # 전체 내용 로드 (약 20k자 → 문제없음)

raw_faq_list = full_text.split('\n\n')

safe_contexts = [[faq.strip()] for faq in raw_faq_list if len(faq.strip()) > 30]

In [6]:
print("chunks 타입:", type(safe_contexts))
print("chunks 길이:", len(safe_contexts))
print("첫 번째 chunk 타입:", type(safe_contexts[0]) if safe_contexts else "빈 리스트")
print("첫 번째 chunk 길이:", len(safe_contexts[0]) if safe_contexts else 0)
print("첫 번째 chunk 미리보기:", safe_contexts[0][:100] if safe_contexts else "없음")

chunks 타입: <class 'list'>
chunks 길이: 126
첫 번째 chunk 타입: <class 'list'>
첫 번째 chunk 길이: 1
첫 번째 chunk 미리보기: ['Q: 회원증을 대리발급 할 수 있나요?\nA: 회원증 대리발급은 만14세 미만 아동, 만65세 이상 어르신, 장애인, 임산부만 가능하며 아래 구비서류를 지참하여 대리인이 방문 시 대리발급이 가능합니다.']


In [7]:
from deepeval.synthesizer import Synthesizer

synthesizer = Synthesizer(
    model=local_llm
)

goldens = synthesizer.generate_goldens_from_contexts(
    contexts=safe_contexts,                   # 전체를 하나의 context로
    include_expected_output=True,
    max_goldens_per_context=3,              
)

Output()

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/usr/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0xeea37f79e2c0> is already entered

In [10]:
# 생성된 golden 리스트 확인
print("생성된 golden 개수:", len(synthesiz
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/usr/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0xeea37f79e2c0> is already entereder.synthetic_goldens))

# 첫 번째 golden 내용 전체 보기
if synthesizer.synthetic_goldens:
    first_golden = synthesizer.synthetic_goldens[0]
    print("\n=== 첫 번째 Golden 상세 ===")
    print("Input (질문):", first_golden.input)
    print("Expected Output (예상 답변):", first_golden.expected_output)
    print("Context (참조 문맥):", first_golden.context)  # 이제 list[str]로 나와야 함
    print("Source File:", first_golden.source_file)
    print("Additional Metadata:", first_golden.additional_metadata)
else:
    print("아직 golden이 생성되지 않았습니다.")

생성된 golden 개수: 27

=== 첫 번째 Golden 상세 ===
Input (질문): 무인예약대출은 신원, 화정어린이, 행신어린이, 마두, 식사, 덕이, 주엽어린이도서관에서만 이용 가능하며, 이들 중 어느 곳의 서비스를 이용할 수 있나?
Expected Output (예상 답변): 무인예약대출 서비스는 신원, 화정어린이, 행신어린이, 마두, 식사, 덕이, 주엽어린이도서관에서 모두 이용 가능합니다. 특정 도서관을 선택하실 경우, 개인의 편의나 위치에 따라 어느 곳을 이용하셔도 됩니다.
Context (참조 문맥): ['- 신청권수 및 대출기간 : 1인 2권 14일 (7일 연장 가능)\n- 신청시간 : 당일 15시까지 신청된 도서를 취합하여 기기에 투입(휴관일 신청도서의 경우 개관일에 투입)\n- 투입시간 : 16시까지 기기투입 -> 신청자에게 알림톡(SMS) 자동 전송 (알림톡 수신 후 방문)\n- 보관기간 : 도서 투입 후 3일간\n- 신청자의 도서관 회원증으로 대출 가능\n- 이용제한 : 6권 미대출시 1개월간 이용정지\n- 도서관 홈페이지(모바일 포함), 리브로피아를 통한 신청 접수\n- 무인예약서비스 운영 도서관 : 신원, 화정어린이, 행신어린이, 마두, 식사, 덕이, 주엽어린이도서관\n\n홈페이지로그인 →도서검색→무인예약대출신청→도서투입 및 SMS수신→도서대출→도서반납\n순으로 이루어집니다."\n\nQ: 지하철역 아무 곳에서나 반납 가능한가요?\nA: 고양시 11개 지하철역에 설치된 무인반납함에 도서 반납 가능합니다._x000D_\n>(대화역, 마두역, 백마역, 백석역, 삼송역, 원당역, 일산역, 주엽역, 지축역, 행신역, 화정역)\n_x000D_\n※ 별꿈도서관 및 작은도서관, 스마트도서관 제외_x000D_\n※ DVD, 책꾸러미 제외_x000D_\n※ 투입한 당일에 즉시 수거 및 반납되지 않으며 반납완료에 1~4일 소요될 수 있음_x000D_\n※ 반납완료 전 연체 발생할 수 있으므로, 즉시 도서대출이 필요할 경우 시립도서관에 직접 반납\

In [11]:
import os

save_dir = "./synthetic_data"
os.makedirs(save_dir, exist_ok=True)

# JSON으로 저장 (가장 추천)
synthesizer.save_as(file_type='json', directory=save_dir)
print(f"JSON 저장 완료: {save_dir}/synthetic_goldens.json")

# 또는 CSV로 저장 (필요시)
synthesizer.save_as(file_type='csv', directory=save_dir)
print(f"CSV 저장 완료: {save_dir}/synthetic_goldens.csv")

Synthetic goldens saved at ./synthetic_data/20260303_212430.json!

JSON 저장 완료: ./synthetic_data/synthetic_goldens.json


Synthetic goldens saved at ./synthetic_data/20260303_212430.csv!

CSV 저장 완료: ./synthetic_data/synthetic_goldens.csv


In [43]:
# MultiContext, Reasoning 질의-응답 생성

In [1]:
import inspect
from deepeval.models import DeepEvalBaseLLM

# 어떤 메서드를 반드시 구현해야 하는지
abstract_methods = [
    name for name, method in inspect.getmembers(DeepEvalBaseLLM)
    if getattr(method, '__isabstractmethod__', False)
]
print("추상 메서드 (반드시 구현):", abstract_methods)

# 전체 메서드 목록 (상속된 것 포함)
all_methods = [name for name, _ in inspect.getmembers(DeepEvalBaseLLM, predicate=inspect.isfunction)]
print("전체 메서드:", all_methods)

추상 메서드 (반드시 구현): ['a_generate', 'generate', 'get_model_name', 'load_model']
전체 메서드: ['__init__', 'a_generate', 'batch_generate', 'generate', 'get_model_name', 'load_model']


In [2]:
import re
import json
import logging
from deepeval.models import DeepEvalBaseLLM, DeepEvalBaseEmbeddingModel
from typing import List
from openai import OpenAI, AsyncOpenAI
import nest_asyncio
nest_asyncio.apply()

logger = logging.getLogger(__name__)  # ✅ 여기서 선언

class VLLMModel(DeepEvalBaseLLM):
    def __init__(self, model_name: str, base_url: str):
        self.model_name = model_name
        self.client = OpenAI(base_url=base_url, api_key="vllm-dummy")
        self.async_client = AsyncOpenAI(base_url=base_url, api_key="vllm-dummy")

    def load_model(self): return self.client
    def get_model_name(self): return self.model_name

    def _is_response_schema(self, schema) -> bool:
        if schema is None:
            return False
        try:
            fields = schema.model_fields if hasattr(schema, 'model_fields') else {}
            return list(fields.keys()) == ['response']
        except Exception:
            return False

    def _get_system_prompt(self, schema) -> str:
        base = "모든 답변은 반드시 한국어로 작성하세요. "
        if schema is not None and self._is_response_schema(schema):
            return base + "간결하고 직접적으로 답변하세요."
        elif schema is not None:
            return base + "반드시 유효한 JSON 형식으로만 응답하세요. 설명이나 마크다운을 포함하지 마세요."
        else:
            return base + "도움이 되는 어시스턴트입니다."

    def _safe_json_parse(self, content: str, schema=None):
        if schema is not None and self._is_response_schema(schema):
            try:
                match = re.search(r'\{.*\}', content, re.DOTALL)
                if match:
                    data = json.loads(match.group(0))
                    if 'response' in data:
                        return schema(response=str(data['response']))
            except Exception:
                pass
            clean_text = content.strip().strip('"').strip("'")
            return schema(response=clean_text)

        try:
            match = re.search(r"(\{.*\}|\[.*\])", content, re.DOTALL)
            if not match:
                raise ValueError("JSON 구조를 찾을 수 없습니다.")

            json_str = match.group(1).strip()
            open_braces = json_str.count('{')
            close_braces = json_str.count('}')
            if open_braces > close_braces:
                json_str += '}' * (open_braces - close_braces)
                json_str = json_str.rstrip()
                if json_str.endswith(','): json_str = json_str[:-1]
                if ']' not in json_str and '[' in json_str: json_str += ']}' 

            data = json.loads(json_str)
            if schema:
                return schema(**data)
            return data

        except Exception as e:
            logger.warning(f"파싱 실패: {e} | 응답 요약: {content[:50]}...")
            if schema:
                try: return schema(data=[])
                except: return None
            return {"data": []}

    def generate(self, prompt: str, schema=None, **kwargs):
        res = self.client.chat.completions.create(
            model=self.model_name,
            messages=[
                {"role": "system", "content": self._get_system_prompt(schema)},  # ✅
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            max_tokens=3000
        )
        return self._safe_json_parse(res.choices[0].message.content, schema)

    async def a_generate(self, prompt: str, schema=None, **kwargs):
        res = await self.async_client.chat.completions.create(
            model=self.model_name,
            messages=[
                {"role": "system", "content": self._get_system_prompt(schema)},  # ✅
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            max_tokens=3000
        )
        return self._safe_json_parse(res.choices[0].message.content, schema)


class VLLMEmbedding(DeepEvalBaseEmbeddingModel):
    def __init__(self, model_name: str, base_url: str):
        self.model_name = model_name
        self.client = OpenAI(base_url=base_url, api_key="vllm-dummy")
        self.async_client = AsyncOpenAI(base_url=base_url, api_key="vllm-dummy")  # ✅ 추가

    def load_model(self): return self.client
    def get_model_name(self): return self.model_name

    def embed_text(self, text: str) -> List[float]:
        resp = self.client.embeddings.create(input=text, model=self.model_name)
        return resp.data[0].embedding

    def embed_texts(self, texts: List[str]) -> List[List[float]]:
        resp = self.client.embeddings.create(input=texts, model=self.model_name)
        return [d.embedding for d in resp.data]

    async def a_embed_text(self, text: str) -> List[float]:  # ✅ 수정
        resp = await self.async_client.embeddings.create(
            input=text, model=self.model_name
        )
        return resp.data[0].embedding

    async def a_embed_texts(self, texts: List[str]) -> List[List[float]]:  # ✅ 수정
        resp = await self.async_client.embeddings.create(
            input=texts, model=self.model_name
        )
        return [d.embedding for d in resp.data]

In [3]:
import pandas as pd

XLSX_PATH = "/home/vsc/LLM_TUNE/QA-FineTune/main/data/고양시도서관 FAQ1.xlsx"
TXT_PATH  = "/home/vsc/LLM_TUNE/QA-FineTune/main/data/고양시도서관_FAQ1.txt"

df = pd.read_excel(XLSX_PATH)
print("컬럼:", df.columns.tolist())
print(f"행 수: {len(df)}")
# df.head(3)   # 주피터라면 표시

# 실제 컬럼명으로 바꾸세요 (당신의 파일 기준으로)
TITLE = df.columns[1]   # 예: '질문' 또는 '제목'
DES   = df.columns[2]   # 예: '답변' 또는 '내용'

with open(TXT_PATH, "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        q = str(row[TITLE]).strip()
        a = str(row[DES]).strip()
        if q and a and q != "nan" and a != "nan":
            f.write(f"Q: {q}\nA: {a}\n\n")

# 파일 확인
with open(TXT_PATH, encoding="utf-8") as f:
    content = f.read()
print(f"저장 완료: {TXT_PATH}")
print(f"파일 크기: {len(content)} 글자")
print("-" * 40)
print(content[:400])

컬럼: ['FAQ', 'TITLE', 'DES']
행 수: 110
저장 완료: /home/vsc/LLM_TUNE/QA-FineTune/main/data/고양시도서관_FAQ1.txt
파일 크기: 20562 글자
----------------------------------------
Q: 회원증을 대리발급 할 수 있나요?
A: 회원증 대리발급은 만14세 미만 아동, 만65세 이상 어르신, 장애인, 임산부만 가능하며 아래 구비서류를 지참하여 대리인이 방문 시 대리발급이 가능합니다.

· 대상 : 만14세 미만 아동, 만65세 이상 어르신, 장애인, 임산부
· 구비서류
  - 공통 : ①위임자 신분증, ②피위임자(대리인) 신분증
  - 장애인 : 장애인 복지카드 또는 장애인 증명서
  - 임신부 : 산모수첩 / 산모 : 주민등록등본(출산 후 12개월까지)
· 방법 : 홈페이지 회원가입 후 위 항목의 해당하는 구비서류를 지참하여 피위임자(대리인)이 도서관 방문

Q: 가족회원이 무엇인가요?
A: 가족회원이란, 고양시 도서관 정회원으로 가입 후 가족회원으로 등록 시 가족명의로 도서대출·도서예약


In [4]:
import logging
import sys

# 모든 디버그 로그를 화면에 출력하도록 설정
logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)

# DeepEval 관련 로거만 따로 레벨 조정 (선택 사항)
logger = logging.getLogger('deepeval')
logger.setLevel(logging.DEBUG)

In [5]:
with open(TXT_PATH, encoding="utf-8") as f:
    full_text = f.read().strip()  # 전체 내용 로드 (약 20k자 → 문제없음)

In [8]:
# 1. 원문 데이터를 FAQ 한 쌍 단위로 쪼개기
# 보통 FAQ는 '\n\n'이나 'Q:'로 구분됩니다. 
raw_faq_list = full_text.split('\n\n')

safe_contexts = [[faq.strip()] for faq in raw_faq_list if len(faq.strip()) > 30]

print(f"총 {len(safe_contexts)}개의 개별 FAQ 문맥이 준비되었습니다.")
print("샘플 예시:", safe_contexts[0])

총 126개의 개별 FAQ 문맥이 준비되었습니다.
샘플 예시: ['Q: 회원증을 대리발급 할 수 있나요?\nA: 회원증 대리발급은 만14세 미만 아동, 만65세 이상 어르신, 장애인, 임산부만 가능하며 아래 구비서류를 지참하여 대리인이 방문 시 대리발급이 가능합니다.']


In [9]:
# ────────────────────────────────────────────────
# 인스턴스 생성 (당신의 vLLM 포트 그대로)
local_llm = VLLMModel(
    model_name="/models/Exaone-3.5-32B-Instruct",
    base_url="http://localhost:8002/v1"
)

local_embedder = VLLMEmbedding(
    model_name="/embeddings/dragonkue/snowflake-arctic-embed-l-v2.0-ko",
    base_url="http://localhost:8003/v1"
)

In [17]:
from deepeval.synthesizer import Synthesizer, Evolution
from deepeval.synthesizer.config import EvolutionConfig

# Multi-hop 질문을 강제하기 위해 MULTICONTEXT 비중을 높입니다.
multi_hop_config = EvolutionConfig(
    evolutions={
        Evolution.CONCRETIZING: 0.5, # 50% 확률로 구체적인 개념 물어봄
        Evolution.REASONING: 0.3,    # 30% 확률로 논리 추론 질문 생성
        Evolution.HYPOTHETICAL: 0.2   # 20% 확률로 가상의 질문 물어봄
    }
)

synthesizer = Synthesizer(
    model=local_llm,
    evolution_config=multi_hop_config
)

# 생성 시작
goldens = synthesizer.generate_goldens_from_contexts(
    contexts=safe_contexts,
    include_expected_output=True,
    max_goldens_per_context=1
)

Output()

2026-03-04 11:34:35,390 [DEBUG] Request options: {'method': 'post', 'url': '/chat/completions', 'files': None, 'idempotency_key': 'stainless-python-retry-3ba4851c-1df0-44e1-930c-242405558536', 'json_data': {'messages': [{'role': 'system', 'content': '모든 답변은 반드시 한국어로 작성하세요. 반드시 유효한 JSON 형식으로만 응답하세요. 설명이나 마크다운을 포함하지 마세요.'}, {'role': 'user', 'content': 'I want you act as a copywriter. Based on the given context, which is list of strings, please generate a list of JSON objects with a `input` key.\n        The `input` can either be a question or a statement that can be addressed by the given context.\n\n        **\n        IMPORTANT: Please make sure to only return in JSON format, with the \'data\' key as a list of JSON objects.\n        You MUST TRY to generate 1 data points, unless the `input` is getting repetitive.\n\n        Example context: ["Einstein won the Nobel Prize for his discovery of penicillin.", "Einstein won the Nobel Prize in 1968."]\n        Example max goldens per context

AttributeError: 'NoneType' object has no attribute 'rewritten_input'

2026-03-04 11:37:23,034 [DEBUG] send_request_headers.started request=<Request [b'POST']>
2026-03-04 11:37:23,034 [DEBUG] send_request_headers.complete
2026-03-04 11:37:23,034 [DEBUG] send_request_body.started request=<Request [b'POST']>
2026-03-04 11:37:23,035 [DEBUG] send_request_body.complete
2026-03-04 11:37:23,035 [DEBUG] receive_response_headers.started request=<Request [b'POST']>
2026-03-04 11:37:35,159 [DEBUG] receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'date', b'Wed, 04 Mar 2026 02:37:22 GMT'), (b'server', b'uvicorn'), (b'content-length', b'741'), (b'content-type', b'application/json')])
2026-03-04 11:37:35,161 [INFO] HTTP Request: POST http://localhost:8002/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-04 11:37:35,162 [DEBUG] receive_response_body.started request=<Request [b'POST']>
2026-03-04 11:37:35,162 [DEBUG] receive_response_body.complete
2026-03-04 11:37:35,163 [DEBUG] response_closed.started
2026-03-04 11:37:35,163 [DEBUG] response_close

In [ ]:
for i, g in enumerate(synthesizer.synthetic_goldens[:3]):
    print(f"[{i}] additional_metadata:", g.additional_metadata)

In [16]:
# 생성된 golden 리스트 확인
print("생성된 golden 개수:", len(synthesizer.synthetic_goldens))

# 첫 번째 golden 내용 전체 보기
if synthesizer.synthetic_goldens:
    first_golden = synthesizer.synthetic_goldens[4]
    print("\n=== 첫 번째 Golden 상세 ===")
    print("Input (질문):", first_golden.input)
    print("Expected Output (예상 답변):", first_golden.expected_output)
    print("Context (참조 문맥):", first_golden.context)  # 이제 list[str]로 나와야 함
    print("Source File:", first_golden.source_file)
    print("Additional Metadata:", first_golden.additional_metadata)
else:
    print("아직 golden이 생성되지 않았습니다.")

생성된 golden 개수: 0
아직 golden이 생성되지 않았습니다.


In [ ]:
import copy

goldens_backup = copy.deepcopy(synthesizer.synthetic_goldens)

# 확인
print(len(goldens_backup))
print(goldens_backup[0].additional_metadata)

In [ ]:
import os
import json
from datetime import datetime

def save_goldens_with_metadata(goldens, directory: str, file_name: str, evolution_config_name: str = "default"):
    os.makedirs(directory, exist_ok=True)
    
    output = []
    for i, g in enumerate(goldens):
        context_text = " ".join(g.context) if g.context else ""
        expected = g.expected_output or ""
        
        # 기존 메타데이터 유지하면서 추가
        existing_metadata = g.additional_metadata or {}
        
        output.append({
            "input": g.input,
            "actual_output": g.actual_output,
            "expected_output": g.expected_output,
            "context": g.context,
            "source_file": g.source_file,
        })
    
    full_path = os.path.join(directory, f"{file_name}.json")
    with open(full_path, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=4)
    
    print(f"💾 저장 완료: {full_path} ({len(output)}개)")
    return full_path

save_goldens_with_metadata(
    goldens=synthesizer.synthetic_goldens,
    directory="./synthetic_data",
    file_name="goyang_library_qa_reasoning_and_multi_hop_2",
    evolution_config_name="CONCRETIZING0.4_REASONING0.4_HYPOTHETICAL0.2"
)